# Stylepal Agent Evaluation with Ragas

Evaluate the Stylepal Stylist Agent using Ragas metrics:
- **Agent Goal Accuracy**: Did the agent achieve the user's goal?
- **Topic Adherence**: Does the agent stay on topic (wardrobe, styling, outfits)?
- **Tool Call Accuracy**: Did the agent invoke the expected tools with correct arguments?



## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "eval":
    project_root = project_root.parent
backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")
_ = load_dotenv(backend_dir / ".env")  # suppress output

In [2]:
import os
print("LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_PROJECT:", os.getenv("LANGCHAIN_PROJECT"))
print("API key present:", bool(os.getenv("LANGCHAIN_API_KEY")))

LANGCHAIN_TRACING_V2: true
LANGCHAIN_PROJECT: stylepal
API key present: True


In [ ]:
# Run once if needed: !pip install ragas pandas

## Run Agent and Collect Traces

In [3]:
from ragas.messages import ToolCall
from services.agent import plan_outfit, get_stylist_graph
from langchain_core.messages import HumanMessage

AGENT_TEST_QUERIES = [
    {
        "query": "What should I wear for a job interview tomorrow?",
        "reference": "Suggest two professional outfit options from the wardrobe for a job interview.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "job interview professional outfit"}),
            ToolCall(name="get_weather", args={"question": "What should I wear for a job interview tomorrow?"}),
        ],
    },
    {
        "query": "What necklines work for an hourglass body type?",
        "reference": "Explain which necklines flatter an hourglass figure based on style principles.",
        "reference_tool_calls": [
            ToolCall(name="retrieve_style_knowledge", args={"query": "necklines hourglass body type"}),
        ],
    },
    {
        "query": "Outfit for a casual Friday at the office",
        "reference": "Suggest two outfit options suitable for casual Friday from the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "casual Friday office outfit"}),
        ],
    },
    {
        "query": "I moved to NYC",
        "reference": "Update the user's profile with their new location.",
        "reference_tool_calls": [
            ToolCall(name="update_profile", args={"location": "NYC"}),
        ],
    },
    {
        "query": "I bought a navy blazer",
        "reference": "Add the new navy blazer to the user's wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="add_wardrobe_item", args={"name": "navy blazer", "category": "outerwear", "color": "navy"}),
        ],
    },
    {
        "query": "I don't wear item 5 anymore, remove it from my wardrobe",
        "reference": "Soft delete item 5 so it no longer appears in suggestions.",
        "reference_tool_calls": [
            ToolCall(name="deprecate_wardrobe_item", args={"item_id": 5}),
        ],
    },
]

def run_agent_and_get_messages(query: str, thread_id: str | None = None) -> list:
    """Run agent and return messages for Ragas conversion. Use unique thread_id per run to avoid accumulated history."""
    import uuid
    graph = get_stylist_graph()
    tid = thread_id or f"eval-{uuid.uuid4().hex[:8]}"
    config = {
        "configurable": {"thread_id": tid},
        "run_name": "StylepalAgent",
        "tags": ["stylepal", "stylist", "eval"],
        "metadata": {"thread_id": tid, "eval": True},
        "recursion_limit": 12,  # Cap agent->tools->agent cycles; prevents runaway hangs
    }
    result = graph.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config,
    )
    return result["messages"]



/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


In [4]:
# Debug: run agent for first query and inspect tool_calls
# Step 1: Quick direct LLM test (~10-30 sec) - does Gemini return tool_calls?
print("--- Direct LLM test (Gemini + bind_tools + tool_choice=any) ---")
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from services.agent import STYLIST_TOOLS
_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=__import__('os').getenv("GEMINI_API_KEY"))
_llm_tools = _llm.bind_tools(STYLIST_TOOLS, tool_choice="any")
_direct = _llm_tools.invoke([HumanMessage(content="What should I wear for a job interview tomorrow?")])
print("Direct response tool_calls:", getattr(_direct, 'tool_calls', None))
print("Direct response has tool_calls:", bool(getattr(_direct, 'tool_calls', None)))



--- Direct LLM test (Gemini + bind_tools + tool_choice=any) ---
Direct response tool_calls: [{'name': 'retrieve_style_knowledge', 'args': {'query': 'job interview outfit'}, 'id': '863f6c6c-2a33-431e-9d79-dccf196c7aa3', 'type': 'tool_call'}]
Direct response has tool_calls: True


In [ ]:
# Step 2: Full graph (slower - runs tools: DB, RAG, weather API)
# Optional: skip this cell and the direct LLM test above; run eval loop directly.
print("\n--- Full graph invoke ---")
import time
print("Waiting 60s to avoid 429 rate limit...")
time.sleep(60)
import importlib
import services.agent as agent_mod
importlib.reload(agent_mod)
from services.agent import get_stylist_graph
from langchain_core.messages import AIMessage, ToolMessage
_sample = AGENT_TEST_QUERIES[0]
_graph = get_stylist_graph()
_tid = __import__('uuid').uuid4().hex[:8]
# recursion_limit caps agent->tools->agent loops; prevents runaway hangs
_result = _graph.invoke({"messages": [HumanMessage(content=_sample["query"])]}, config={"configurable": {"thread_id": f"eval-{_tid}"}, "metadata": {"eval": True}, "recursion_limit": 15})
_msgs = _result["messages"]
print("Query:", _sample["query"])
print("Total messages:", len(_msgs))
print("ToolMessages:", sum(1 for m in _msgs if isinstance(m, ToolMessage)))
for i, m in enumerate(_msgs):
    print(f"[{i}] {type(m).__name__}")
    if isinstance(m, AIMessage):
        tc = getattr(m, 'tool_calls', None)
        print(f"    tool_calls: {tc}")


--- Full graph invoke ---
Waiting 60s after direct LLM test to avoid 429 rate limit...


/Users/sireeshapulipati/stylepal/backend/services/agent.py:638: UserWarning: The 'config' parameter should be typed as 'RunnableConfig' or 'RunnableConfig | None', not 'dict | None'. 
  


In [4]:
import json
import os
import warnings
warnings.filterwarnings("ignore", message=".*google.generativeai.*", category=FutureWarning)

from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from ragas.integrations.langgraph import convert_to_ragas_messages
from ragas.dataset_schema import MultiTurnSample
from ragas.metrics import AgentGoalAccuracyWithReference, NonLLMStringSimilarity, TopicAdherenceScore, ToolCallAccuracy
from ragas.llms import LangchainLLMWrapper

def ensure_tool_calls_for_ragas(messages):
    """Ragas reads tool_calls from additional_kwargs (OpenAI format). Gemini uses message.tool_calls."""
    out = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tc = m.tool_calls
            if tc:
                def _to_openai(t, i):
                    n = t.get("name", "") if isinstance(t, dict) else getattr(t, "name", "")
                    a = t.get("args", {}) if isinstance(t, dict) else getattr(t, "args", {})
                    a = a if isinstance(a, dict) else {}
                    return {"id": t.get("id", f"call_{i}") if isinstance(t, dict) else getattr(t, "id", f"call_{i}"), "type": "function", "function": {"name": n, "arguments": json.dumps(a)}}
                openai_tc = [_to_openai(t, i) for i, t in enumerate(tc)]
                kwargs = dict(getattr(m, "additional_kwargs", None) or {})
                kwargs["tool_calls"] = openai_tc
                out.append(AIMessage(content=m.content, additional_kwargs=kwargs))
                continue
        out.append(m)
    return out

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
)

REFERENCE_TOPICS = ["wardrobe", "styling", "outfit suggestions", "fashion", "professional dress", "clothing"]

scores_goal = []
scores_topic = []
scores_tool_call = []



/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_82885/605291030.py:10: DeprecationWarning: Importing AgentGoalAccuracyWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AgentGoalAccuracyWithReference
  from ragas.metrics import AgentGoalAccuracyWithReference, NonLLMStringSimilarity, TopicAdherenceScore, ToolCallAccuracy
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_82885/605291030.py:10: DeprecationWarning: Importing NonLLMStringSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMStringSimilarity
  from ragas.metrics import AgentGoalAccuracyWithReference, NonLLMStringSimilarity, TopicAdherenceScore, ToolCallAccuracy
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_82885/605291030.py:10: DeprecationWarning: Impo

In [5]:
import importlib
import services.agent as agent_mod
import services.wardrobe_service
importlib.reload(services.wardrobe_service)
importlib.reload(agent_mod)
from services.agent import get_stylist_graph

In [5]:
import time
EVAL_DELAY_SEC = 60  # Wait between evals to avoid 429 rate limit
EVAL_SAMPLES = AGENT_TEST_QUERIES  # Use AGENT_TEST_QUERIES[:2] for quick test (2 samples)

for i, sample in enumerate(EVAL_SAMPLES):
    if i > 0:
        print(f"Waiting {EVAL_DELAY_SEC}s before next eval...")
        time.sleep(EVAL_DELAY_SEC)
    print(f"Eval {i+1}/{len(EVAL_SAMPLES)}: {sample['query'][:50]}...", flush=True)
    print("  Running agent...", flush=True)
    t0 = time.perf_counter()
    messages = run_agent_and_get_messages(sample["query"])
    t_agent = time.perf_counter() - t0
    print(f"  Agent done in {t_agent:.1f}s. Running Ragas scorers...", flush=True)
    messages = ensure_tool_calls_for_ragas(messages)
    ragas_trace = convert_to_ragas_messages(messages)
    
    mt_sample = MultiTurnSample(
        user_input=ragas_trace,
        reference=sample["reference"],
    )
    
    goal_scorer = AgentGoalAccuracyWithReference()
    goal_scorer.llm = evaluator_llm
    t1 = time.perf_counter()
    goal_score = await goal_scorer.multi_turn_ascore(mt_sample)
    print(f"    Goal scorer: {time.perf_counter() - t1:.1f}s", flush=True)
    scores_goal.append(goal_score)
    
    topic_scorer = TopicAdherenceScore(llm=evaluator_llm, mode="precision")
    mt_sample_ta = MultiTurnSample(user_input=ragas_trace, reference_topics=REFERENCE_TOPICS)
    t2 = time.perf_counter()
    topic_score = await topic_scorer.multi_turn_ascore(mt_sample_ta)
    print(f"    Topic scorer: {time.perf_counter() - t2:.1f}s", flush=True)
    scores_topic.append(topic_score)
    
    tool_call_scorer = ToolCallAccuracy(strict_order=False, arg_comparison_metric=NonLLMStringSimilarity())  # Fuzzy match for RAG query args
    mt_sample_tc = MultiTurnSample(
        user_input=ragas_trace,
        reference_tool_calls=sample.get("reference_tool_calls", []),
    )
    t3 = time.perf_counter()
    tool_call_score = await tool_call_scorer.multi_turn_ascore(mt_sample_tc)
    print(f"    Tool-call scorer: {time.perf_counter() - t3:.1f}s", flush=True)
    scores_tool_call.append(tool_call_score)
    
    t_total = time.perf_counter() - t0
    print(f"  Agent Goal: {goal_score:.4f} | Topic: {topic_score:.4f} | Tool Call: {tool_call_score:.4f} | Total: {t_total:.1f}s")
    print()

Eval 1/6: What should I wear for a job interview tomorrow?...
  Running agent...


/Users/sireeshapulipati/stylepal/backend/services/agent.py:649: UserWarning: The 'config' parameter should be typed as 'RunnableConfig' or 'RunnableConfig | None', not 'dict | None'. 
  builder.add_node("agent", agent)


  Agent done in 72.5s. Running Ragas scorers...
    Goal scorer: 5.5s
    Topic scorer: 14.6s
    Tool-call scorer: 0.0s
  Agent Goal: 0.0000 | Topic: 0.3333 | Tool Call: 0.0000 | Total: 92.7s

Waiting 60s before next eval...


/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (15) vs reference tool calls (3). Only the first 3 tool calls will be compared.
  warnings.warn(


Eval 2/6: What necklines work for an hourglass body type?...
  Running agent...
  Agent done in 27.5s. Running Ragas scorers...
    Goal scorer: 4.2s
    Topic scorer: 14.0s
    Tool-call scorer: 0.0s
  Agent Goal: 1.0000 | Topic: 0.0000 | Tool Call: 0.0000 | Total: 45.7s

Waiting 60s before next eval...


/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (5) vs reference tool calls (1). Only the first 1 tool calls will be compared.
  warnings.warn(


Eval 3/6: Outfit for a casual Friday at the office...
  Running agent...
  Agent done in 37.0s. Running Ragas scorers...
    Goal scorer: 6.7s
    Topic scorer: 12.3s
    Tool-call scorer: 0.0s
  Agent Goal: 0.0000 | Topic: 0.3333 | Tool Call: 0.0000 | Total: 56.0s

Waiting 60s before next eval...


/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (15) vs reference tool calls (2). Only the first 2 tool calls will be compared.
  warnings.warn(


Eval 4/6: I moved to NYC...
  Running agent...
  Agent done in 64.3s. Running Ragas scorers...
    Goal scorer: 4.4s
    Topic scorer: 8.4s
    Tool-call scorer: 0.0s
  Agent Goal: 0.0000 | Topic: 0.4000 | Tool Call: 0.0000 | Total: 77.1s

Waiting 60s before next eval...
Eval 5/6: I bought a navy blazer...
  Running agent...
  Agent done in 30.5s. Running Ragas scorers...
    Goal scorer: 4.3s
    Topic scorer: 5.6s
    Tool-call scorer: 0.0s
  Agent Goal: 0.0000 | Topic: 0.6667 | Tool Call: 0.0000 | Total: 40.3s

Waiting 60s before next eval...
Eval 6/6: I don't wear item 5 anymore, remove it from my war...
  Running agent...
  Agent done in 21.3s. Running Ragas scorers...
    Goal scorer: 4.4s
    Topic scorer: 11.2s
    Tool-call scorer: 0.0s
  Agent Goal: 1.0000 | Topic: 0.7143 | Tool Call: 0.0000 | Total: 36.9s



/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (6) vs reference tool calls (1). Only the first 1 tool calls will be compared.
  warnings.warn(


In [6]:
import numpy as np
print("Agent Evaluation Summary:")
print(f"  Mean Agent Goal Accuracy: {np.mean(scores_goal):.4f}")
print(f"  Mean Topic Adherence: {np.mean(scores_topic):.4f}")
print(f"  Mean Tool Call Accuracy: {np.mean(scores_tool_call):.4f}")

Agent Evaluation Summary:
  Mean Agent Goal Accuracy: 0.3333
  Mean Topic Adherence: 0.4079
  Mean Tool Call Accuracy: 0.0000


## Extended Test Cases (single-turn, full context)

Additional evaluation pairs where the user input includes all needed data—no back-and-forth required.

In [7]:
AGENT_TEST_QUERIES_EXTENDED = [
    # Single-day / single-event
    {
        "query": "What to wear for a dinner date this Saturday?",
        "reference": "Suggest two outfit options suitable for a dinner date from the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "dinner date outfit"}),
            ToolCall(name="get_weather", args={"question": "What to wear for a dinner date this Saturday?"}),
        ],
    },
    {
        "query": "Interview outfit for next Tuesday",
        "reference": "Suggest two professional outfit options for a job interview.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "job interview professional outfit"}),
        ],
    },
    # Multi-day trips (full context)
    {
        "query": "Plan outfits for my 3-day trip to Paris July 15-17: Day 1 travel, Day 2 museum and casual dinner, Day 3 business meeting",
        "reference": "Suggest one primary outfit per day/occasion, optionally 1-3 extra items to pack.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "Paris July 15-17", "location": "Paris"}),
        ],
    },
    {
        "query": "5-day business trip to NYC Mon-Fri: meetings all day, one formal dinner Wednesday evening",
        "reference": "Suggest one outfit per day/occasion for the business trip.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "NYC weather forecast", "location": "NYC"}),
        ],
    },
    # Conferences (full context)
    {
        "query": "Outfit for a local tech conference I'm attending next Tuesday, business casual",
        "reference": "Suggest two outfit options suitable for a tech conference.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "business casual conference outfit"}),
            ToolCall(name="get_weather", args={"question": "weather next Tuesday"}),
        ],
    },
    # Profile updates
    {
        "query": "I prefer tailored fits now",
        "reference": "Update the user's profile with their new style preferences.",
        "reference_tool_calls": [
            ToolCall(name="update_profile", args={"silhouette_preferences": "tailored"}),
        ],
    },
    {
        "query": "I'm 35 and my body type is more pear-shaped",
        "reference": "Update the user's profile with age and body type.",
        "reference_tool_calls": [
            ToolCall(name="update_profile", args={"age": 35, "body_type": "pear"}),
        ],
    },
    # Add wardrobe item (full context)
    {
        "query": "I bought a navy blazer from Ralph Lauren last month for work",
        "reference": "Add the navy blazer to the wardrobe with brand, purchase date, and occasion.",
        "reference_tool_calls": [
            ToolCall(name="add_wardrobe_item", args={"name": "navy blazer", "category": "outerwear", "color": "navy", "brand": "Ralph Lauren", "purchased_at": "last month", "occasion_tags": "work"}),
        ],
    },
    {
        "query": "Add my new white sneakers from Nike, bought in March 2024, casual wear",
        "reference": "Add the white sneakers to the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="add_wardrobe_item", args={"name": "white sneakers", "category": "shoes", "color": "white", "brand": "Nike", "purchased_at": "March 2024", "occasion_tags": "casual"}),
        ],
    },
    # Deprecate item (with confirmed id)
    {
        "query": "Remove item 12 from my wardrobe",
        "reference": "Soft delete item 12 so it no longer appears in suggestions.",
        "reference_tool_calls": [
            ToolCall(name="deprecate_wardrobe_item", args={"item_id": 12}),
        ],
    },
    # Informational
    {
        "query": "What colors pair well with navy?",
        "reference": "Explain which colors complement navy based on style principles.",
        "reference_tool_calls": [
            ToolCall(name="retrieve_style_knowledge", args={"query": "colors pair navy"}),
        ],
    },
    {
        "query": "What's the golden mean in fashion?",
        "reference": "Explain the golden mean concept in fashion.",
        "reference_tool_calls": [
            ToolCall(name="retrieve_style_knowledge", args={"query": "golden mean fashion proportions"}),
        ],
    },
    # Weather-aware
    {
        "query": "Outfit for tomorrow, it might rain",
        "reference": "Suggest outfit options considering rain, with weather check.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "tomorrow weather rain"}),
        ],
    },
    {
        "query": "What to wear in Paris next week?",
        "reference": "Suggest outfit options for Paris with weather consideration.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "Paris next week", "location": "Paris"}),
        ],
    },
]

In [8]:
# Run evaluation on extended test cases (uses same evaluator setup from cell 6)
scores_goal_ext = []
scores_topic_ext = []
scores_tool_call_ext = []

for i, sample in enumerate(AGENT_TEST_QUERIES_EXTENDED):
    if i > 0:
        print(f"Waiting {EVAL_DELAY_SEC}s before next eval...")
        time.sleep(EVAL_DELAY_SEC)
    print(f"Eval {i+1}/{len(AGENT_TEST_QUERIES_EXTENDED)}: {sample['query'][:50]}...")
    t0 = time.perf_counter()
    messages = run_agent_and_get_messages(sample["query"])
    t_agent = time.perf_counter() - t0
    print(f"  Agent done in {t_agent:.1f}s. Running Ragas scorers...", flush=True)
    messages = ensure_tool_calls_for_ragas(messages)
    ragas_trace = convert_to_ragas_messages(messages)
    
    goal_scorer = AgentGoalAccuracyWithReference()
    goal_scorer.llm = evaluator_llm
    topic_scorer = TopicAdherenceScore(llm=evaluator_llm, mode="precision")
    tool_call_scorer = ToolCallAccuracy(strict_order=False, arg_comparison_metric=NonLLMStringSimilarity())
    
    mt_sample = MultiTurnSample(user_input=ragas_trace, reference=sample["reference"])
    t1 = time.perf_counter()
    goal_score = await goal_scorer.multi_turn_ascore(mt_sample)
    print(f"    Goal scorer: {time.perf_counter() - t1:.1f}s", flush=True)
    scores_goal_ext.append(goal_score)
    
    mt_sample_ta = MultiTurnSample(user_input=ragas_trace, reference_topics=REFERENCE_TOPICS)
    t2 = time.perf_counter()
    topic_score = await topic_scorer.multi_turn_ascore(mt_sample_ta)
    print(f"    Topic scorer: {time.perf_counter() - t2:.1f}s", flush=True)
    scores_topic_ext.append(topic_score)
    
    mt_sample_tc = MultiTurnSample(user_input=ragas_trace, reference_tool_calls=sample.get("reference_tool_calls", []))
    t3 = time.perf_counter()
    tool_call_score = await tool_call_scorer.multi_turn_ascore(mt_sample_tc)
    print(f"    Tool-call scorer: {time.perf_counter() - t3:.1f}s", flush=True)
    scores_tool_call_ext.append(tool_call_score)
    
    t_total = time.perf_counter() - t0
    print(f"Query: {sample['query'][:60]}...")
    print(f"  Agent Goal: {goal_score:.4f} | Topic: {topic_score:.4f} | Tool Call: {tool_call_score:.4f} | Total: {t_total:.1f}s")
    print()

print("Extended Evaluation Summary:")
print(f"  Mean Agent Goal Accuracy: {np.mean(scores_goal_ext):.4f}")
print(f"  Mean Topic Adherence: {np.mean(scores_topic_ext):.4f}")
print(f"  Mean Tool Call Accuracy: {np.mean(scores_tool_call_ext):.4f}")

Eval 1/14: What to wear for a dinner date this Saturday?...
  Agent done in 40.0s. Running Ragas scorers...
    Goal scorer: 4.3s
    Topic scorer: 7.2s
    Tool-call scorer: 0.0s
Query: What to wear for a dinner date this Saturday?...
  Agent Goal: 0.0000 | Topic: 0.6667 | Tool Call: 0.0000 | Total: 51.6s

Waiting 60s before next eval...
Eval 2/14: Interview outfit for next Tuesday...
  Agent done in 34.2s. Running Ragas scorers...
    Goal scorer: 6.7s
    Topic scorer: 14.3s
    Tool-call scorer: 0.0s
Query: Interview outfit for next Tuesday...
  Agent Goal: 0.0000 | Topic: 0.5000 | Tool Call: 0.0000 | Total: 55.2s

Waiting 60s before next eval...
Eval 3/14: Plan outfits for my 3-day trip to Paris July 15-17...
  Agent done in 76.9s. Running Ragas scorers...
    Goal scorer: 5.1s
    Topic scorer: 26.5s
    Tool-call scorer: 0.0s
Query: Plan outfits for my 3-day trip to Paris July 15-17: Day 1 tr...
  Agent Goal: 0.0000 | Topic: 0.7143 | Tool Call: 0.0000 | Total: 108.5s

Waiting 60

/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (18) vs reference tool calls (2). Only the first 2 tool calls will be compared.
  warnings.warn(


Eval 4/14: 5-day business trip to NYC Mon-Fri: meetings all d...
  Agent done in 42.5s. Running Ragas scorers...
    Goal scorer: 4.9s
    Topic scorer: 22.0s
    Tool-call scorer: 0.0s
Query: 5-day business trip to NYC Mon-Fri: meetings all day, one fo...
  Agent Goal: 0.0000 | Topic: 0.3750 | Tool Call: 0.0000 | Total: 69.4s

Waiting 60s before next eval...


/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (20) vs reference tool calls (2). Only the first 2 tool calls will be compared.
  warnings.warn(


Eval 5/14: Outfit for a local tech conference I'm attending n...
  Agent done in 131.4s. Running Ragas scorers...
    Goal scorer: 5.1s
    Topic scorer: 21.0s
    Tool-call scorer: 0.0s
Query: Outfit for a local tech conference I'm attending next Tuesda...
  Agent Goal: 0.0000 | Topic: 0.6000 | Tool Call: 0.0000 | Total: 157.5s

Waiting 60s before next eval...


...

Eval 6/14: I prefer tailored fits now...
  Agent done in 0.0s. Running Ragas scorers...
    Goal scorer: 3.3s
    Topic scorer: 7.0s
    Tool-call scorer: 0.0s
Query: I prefer tailored fits now...
  Agent Goal: 0.0000 | Topic: 0.0000 | Tool Call: 0.0000 | Total: 10.4s

Waiting 60s before next eval...


/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:129: UserWarning: No tool calls found in the user input
  warnings.warn("No tool calls found in the user input")


Eval 7/14: I'm 35 and my body type is more pear-shaped...
  Agent done in 119.6s. Running Ragas scorers...
    Goal scorer: 7.0s
    Topic scorer: 9.0s
    Tool-call scorer: 0.0s
Query: I'm 35 and my body type is more pear-shaped...
  Agent Goal: 0.0000 | Topic: 0.2500 | Tool Call: 0.0000 | Total: 135.6s

Waiting 60s before next eval...
Eval 8/14: I bought a navy blazer from Ralph Lauren last mont...
  Agent done in 24.8s. Running Ragas scorers...
    Goal scorer: 5.2s
    Topic scorer: 10.6s
    Tool-call scorer: 0.0s
Query: I bought a navy blazer from Ralph Lauren last month for work...
  Agent Goal: 0.0000 | Topic: 0.5000 | Tool Call: 0.0000 | Total: 40.6s

Waiting 60s before next eval...
Eval 9/14: Add my new white sneakers from Nike, bought in Mar...
  Agent done in 34.1s. Running Ragas scorers...
    Goal scorer: 4.9s
    Topic scorer: 18.7s
    Tool-call scorer: 0.0s
Query: Add my new white sneakers from Nike, bought in March 2024, c...
  Agent Goal: 1.0000 | Topic: 0.4286 | Too

/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/ragas/metrics/_tool_call_accuracy.py:147: UserWarning: Length mismatch: predicted tool calls (7) vs reference tool calls (1). Only the first 1 tool calls will be compared.
  warnings.warn(


Eval 10/14: Remove item 12 from my wardrobe...
  Agent done in 24.8s. Running Ragas scorers...
    Goal scorer: 4.6s
    Topic scorer: 5.6s
    Tool-call scorer: 0.0s
Query: Remove item 12 from my wardrobe...
  Agent Goal: 0.0000 | Topic: 0.3333 | Tool Call: 0.0000 | Total: 34.9s

Waiting 60s before next eval...
Eval 11/14: What colors pair well with navy?...
  Agent done in 40.0s. Running Ragas scorers...
    Goal scorer: 3.8s
    Topic scorer: 11.5s
    Tool-call scorer: 0.0s
Query: What colors pair well with navy?...
  Agent Goal: 0.0000 | Topic: 1.0000 | Tool Call: 0.0000 | Total: 55.3s

Waiting 60s before next eval...
Eval 12/14: What's the golden mean in fashion?...
  Agent done in 27.8s. Running Ragas scorers...
    Goal scorer: 3.2s
    Topic scorer: 56.7s
    Tool-call scorer: 0.0s
Query: What's the golden mean in fashion?...
  Agent Goal: 0.0000 | Topic: 0.0000 | Tool Call: 0.0000 | Total: 87.7s

Waiting 60s before next eval...
Eval 13/14: Outfit for tomorrow, it might rain..

## Combined Conclusions: Original vs Extended Evaluation

### Summary Comparison

| Metric | Original (6 samples) | Extended (14 samples) |
|--------|----------------------|------------------------|
| **Agent Goal Accuracy** | 0.33 | 0.07 |
| **Topic Adherence** | 0.41 | 0.48 |
| **Tool Call Accuracy** | 0.00 | 0.00 |



### 1. Agent Goal Accuracy Drops Sharply on Extended Set

- **Original:** 2/6 passed (necklines, remove item 5).
- **Extended:** 1/14 passed (add white sneakers).
- Extended cases are harder: multi-day trips, richer profile updates, more complex add-wardrobe cases.
- The agent consistently fails on:
  - Outfit suggestions (dinner date, interview, conference, trips)
  - Profile updates ("tailored fits", "pear-shaped")
  - Add wardrobe with extra fields (brand, date, occasion)
  - Deprecate item (remove item 12 scored 0 vs remove item 5 scoring 1 in original)



### 2. Tool Call Accuracy Remains 0 in Both Evaluations

- **Length mismatch:** Agent makes many more tool calls than reference (e.g. 18 vs 2, 20 vs 2).
- Ragas compares only the first N calls; sequence alignment fails when counts differ.
- Two extended cases show **no tool calls**:
  - "I prefer tailored fits now" — agent finished in 0.0s, likely no `update_profile`
  - "I'm 35 and my body type is more pear-shaped" — "No tool calls found in the user input"
- The metric is poorly suited to the agent's multi-step behavior.



### 3. Topic Adherence Improves Slightly (0.41 → 0.48)

- Extended cases cover more topic types.
- High topic scores where goal fails (e.g. "What colors pair with navy?" — Topic 1.0, Goal 0).
- Low topic scores on profile/body-type updates (0.00, 0.25).
- Topic scorer may treat profile/wardrobe updates as less on-topic than outfit/style questions.



### 4. Notable Extended-Case Issues

| Case | Issue |
|------|-------|
| Tailored fits | Agent done in 0.0s — likely no tool use |
| Pear-shaped body | No tool calls found — `update_profile` not called |
| Remove item 12 | Goal 0 vs remove item 5 (original) Goal 1 — inconsistent deprecation behavior |
| Golden mean in fashion | Topic 0 — possibly off-topic or edge case |
| Multi-day trips | Goal 0 — complex planning not meeting reference expectations |



### 5. Overall Assessment

- The agent performs poorly on the extended set: only 1/14 goals met.
- Tool Call Accuracy is not informative with the current metric and reference design.
- Topic Adherence is moderate and somewhat inconsistent across query types.
- **Critical gaps:**
  1. Profile updates often not executed (no tool calls).
  2. Add-wardrobe with rich metadata (brand, date, occasion) mostly fails.
  3. Multi-day trip planning does not meet reference expectations.
  4. Deprecation behavior varies by item ID.



### 6. Recommendations

1. **Agent behavior:** Debug why profile updates ("tailored fits", "pear-shaped") produce no tool calls.
2. **References:** Revisit reference goals for multi-day trips and complex add-wardrobe cases.
3. **Tool call metric:** Use a different evaluation (e.g. subset/coverage of expected tools) instead of strict sequence matching.
4. **Deprecation:** Investigate why remove item 12 fails while remove item 5 passes.